## Initial Domain Filtering and Cleaning

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import re
import glob
import gzip
import warnings
from pathlib import Path
from tqdm import tqdm

# VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Language detection — seeded for reproducibility
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
YELP_BUSINESS_PATH   = "yelp_academic_dataset_business.json"
YELP_REVIEW_PATH     = "yelp_academic_dataset_review.json"
GOOGLE_META_DIR = Path("C:\\Users\\KHLOE\\Documents\\FYP\\mcauley")
GOOGLE_REVIEW_DIR = Path("D:\\.FYP\\mcauley")

OUTPUT_DIR = Path("output/stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Discover state files ───────────────────────────────────────────────────────
google_meta_files   = sorted(GOOGLE_META_DIR.glob("*.json"))
google_review_files = sorted(GOOGLE_REVIEW_DIR.glob("*.json"))

if len(google_meta_files) == 0:
    print("⚠️  No metadata files found — verify GOOGLE_META_DIR path.")
if len(google_review_files) == 0:
    print("⚠️  No review files found — verify GOOGLE_REVIEW_DIR path.")

CHUNK_SIZE = 10000          # rows per chunk — tune down if RAM is tight
MIN_WORD_COUNT = 5
MIN_LOCATIONS  = 100
DATE_START     = "2015-01-01"
DATE_END       = "2018-12-31"

print("✅ Imports and configuration loaded.")
print(f"   Output directory : {OUTPUT_DIR.resolve()}")
print(f"   Chunk size       : {CHUNK_SIZE:,} rows")
print(f"   Min word count   : {MIN_WORD_COUNT}")
print(f"   Min locations    : {MIN_LOCATIONS}")
print(f"   Date window      : {DATE_START} → {DATE_END}")
print(f"   Google metadata files found  : {len(google_meta_files)}")
print(f"   Google review files found    : {len(google_review_files)}")

In [ ]:
# ─────────────── helper for metadata filtration ──────────────────────────────

def word_count(text: str) -> int:
    """Fast whitespace-based word count."""
    return len(text.split())

def normalize_chain_name(name: str) -> str:
    """
    Lowercase, strip punctuation/possessives, collapse whitespace.
    Ensures 'McDonald\\'s', 'Mcdonalds', 'MC DONALDS' all map to 'mcdonalds'.
    """
    name = name.lower()
    name = re.sub(r"['\u2019]s?\b", "", name)   # strip possessives
    name = re.sub(r"[^a-z0-9\s]", " ", name)    # strip non-alphanumeric
    name = re.sub(r"\s+", " ", name).strip()
    return name

def is_restaurant_category(categories) -> bool:
    """
    Returns True if any category token matches restaurant-domain keywords.
    Handles both comma-separated strings (Yelp) and lists (Google).
    """
    RESTAURANT_KEYWORDS = {
        "restaurant", "restaurants"
    }
    if categories is None or (isinstance(categories, float) and np.isnan(categories)):
        return False
    if isinstance(categories, list):
        cat_str = " ".join(str(c).lower() for c in categories)
    else:
        cat_str = str(categories).lower()
    return any(kw in cat_str for kw in RESTAURANT_KEYWORDS)

# ─────────────── helper for Review Data ──────────────────────────────
MISMATCH_HIGH_STAR_THRESH  = 4      # stars >= this value considered "high"
MISMATCH_LOW_STAR_THRESH   = 2      # stars <= this value considered "low"
MISMATCH_NEG_VADER         = -0.5   # VADER below this = strongly negative text
MISMATCH_POS_VADER         =  0.5   # VADER above this = strongly positive text

def detect_language_safe(text: str) -> str:
    """
    Returns ISO 639-1 language code or 'unknown' on failure.
    VADER is fast but text must be short — sample first 300 chars for speed.
    """
    try:
        return detect(text[:300])
    except Exception:
        return "unknown"

analyzer = SentimentIntensityAnalyzer()

def vader_compound(text: str) -> float:
    return analyzer.polarity_scores(text[:512])["compound"]

def _vader_worker(text_series_chunk):
    return text_series_chunk.apply(vader_compound)

def flag_mismatch(stars: float, vader: float) -> bool:
    """
    Returns True if review should be REMOVED due to rating–text mismatch.
    
    Two mismatch cases:
      1. High star (≥4) + strongly negative text (VADER < -0.5)  → sarcasm / misclick
      2. Low star  (≤2) + strongly positive text (VADER >  0.5)  → irony / misclick
    """
    high_star_neg_text = (stars >= MISMATCH_HIGH_STAR_THRESH) and (vader < MISMATCH_NEG_VADER)
    low_star_pos_text  = (stars <= MISMATCH_LOW_STAR_THRESH)  and (vader > MISMATCH_POS_VADER)
    return high_star_neg_text or low_star_pos_text


### Yelp Metadata

In [ ]:
# Cell 4 — PHASE 1A: Load & filter YELP business metadata
# Goal: produce a set of business_ids belonging to US restaurant chains with ≥100 locations

print("=" * 65)
print("PHASE 1A — Yelp Business Metadata: Restaurant Chain Filtering")
print("=" * 65)

yelp_biz_chunks = []

for chunk in tqdm(
    pd.read_json(YELP_BUSINESS_PATH, lines=True, chunksize=CHUNK_SIZE),
    desc="Reading Yelp business"
):
    # Step 1: Keep US businesses only (Yelp Open Dataset includes Canada)
    chunk = chunk[chunk["state"].notna()]
    US_STATES = {
        "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL",
        "IN","IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT",
        "NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI",
        "SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
    }
    chunk = chunk[chunk["state"].isin(US_STATES)]

    # Step 1.5: Keep only open businesses
    chunk = chunk[chunk["is_open"] != 0]

    # Step 2: Keep only restaurant-domain businesses
    chunk = chunk[chunk["categories"].apply(is_restaurant_category)]

    keep = ["business_id", "name", "address", "latitude", "longitude",
            "state", "categories"]
    keep = [c for c in keep if c in chunk.columns]
    yelp_biz_chunks.append(chunk[keep].copy())

yelp_biz = pd.concat(yelp_biz_chunks, ignore_index=True)
print(f"\n   US restaurant businesses loaded : {len(yelp_biz):>10,}")

yelp_biz = yelp_biz[yelp_biz["name"].str.contains(r"[A-Za-z]", na=False)]
print(f"   US restaurant with mostly ASCII characters : {len(yelp_biz):>10,}")

# Step 3: Normalize chain names for grouping
yelp_biz["chain_name_norm"] = yelp_biz["name"].apply(normalize_chain_name)

# Step 4: Count locations per normalized chain name
chain_counts_yelp = yelp_biz.groupby("chain_name_norm")["business_id"].count()
chain_counts_yelp.name = "location_count"

# Step 5: Keep only chains with ≥100 locations
large_chains_yelp = chain_counts_yelp[chain_counts_yelp >= MIN_LOCATIONS]
print(f"   Chains with ≥{MIN_LOCATIONS} Yelp locations : {len(large_chains_yelp):>10,}")

# Step 6: Map back to get qualifying business_ids
yelp_biz_filtered = yelp_biz[yelp_biz["chain_name_norm"].isin(large_chains_yelp.index)].copy()
yelp_biz_filtered = yelp_biz_filtered.merge(
    large_chains_yelp.reset_index(),
    on="chain_name_norm", how="left"
)

YELP_VALID_BIZ_IDS = set(yelp_biz_filtered["business_id"])
print(f"   Qualifying business_ids          : {len(YELP_VALID_BIZ_IDS):>10,}")

# Validation: confirm geo fields are present
geo_coverage = yelp_biz_filtered[["latitude","longitude"]].notna().mean() * 100
print(f"\n   Geo coverage — latitude : {geo_coverage['latitude']:.1f}%")
print(f"   Geo coverage — longitude: {geo_coverage['longitude']:.1f}%")

# Validation: preview top 10 chains by location count
print("\n   ── Top 10 Yelp chains by location count ──")
print(
    yelp_biz_filtered.groupby("chain_name_norm")["location_count"]
    .first()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

# ── CHANGE: save the full business lookup (needed for review join below) ───────
yelp_biz_lookup = yelp_biz_filtered[
    ["business_id", "name", "address", "latitude", "longitude"]
].drop_duplicates("business_id").copy()

print(f"\n✅ Yelp business lookup built — {len(yelp_biz_lookup):,} unique businesses.")

In [ ]:
# Define output filename
output_parquet_path = "yelp_biz_lookup.parquet"

print(f"\n💾 Saving {len(yelp_biz_lookup):,} records to Parquet...")

# Save dataframe
yelp_biz_lookup.to_parquet(
    output_parquet_path,
    engine="pyarrow",  # Fast and reliable engine
    compression="snappy",  # Good balance of speed and size
    index=False
)

print(f"✅ Successfully saved to: {output_parquet_path}")

### Yelp Review Data

In [ ]:
# Cell 5 — PHASE 1B: Load & filter YELP reviews
# Goal: keep only reviews for qualifying business_ids, within date window

print("=" * 65)
print("PHASE 1B — Yelp Reviews: Initial Load & Phase 1 Cleaning")
print("=" * 65)

yelp_review_chunks = []

for chunk in tqdm(
    pd.read_json(YELP_REVIEW_PATH, lines=True, chunksize=CHUNK_SIZE),
    desc="Reading Yelp reviews"
):
    # Filter 1: Only reviews for qualifying businesses
    chunk = chunk[chunk["business_id"].isin(YELP_VALID_BIZ_IDS)]
    if chunk.empty:
        continue

    # Filter 2: Date window
    chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
    chunk = chunk[
        (chunk["date"] >= DATE_START) & (chunk["date"] <= DATE_END)
    ]
    if chunk.empty:
        continue

    # Filter 3: Drop null text
    chunk = chunk[chunk["text"].notna() & (chunk["text"].str.strip() != "")]

    # Filter 4: Lowercase normalization (Phase 1 requirement)
    chunk["text"] = chunk["text"].str.lower().str.strip()

    # Filter 5: Drop reviews with fewer than MIN_WORD_COUNT words
    chunk = chunk[chunk["text"].apply(word_count) >= MIN_WORD_COUNT]

    # Keep only needed columns at this stage
    chunk = chunk[["review_id", "business_id", "stars", "text", "date"]].copy()
    yelp_review_chunks.append(chunk)

yelp_reviews_raw = pd.concat(yelp_review_chunks, ignore_index=True)
print(f"\n   Reviews after domain + date + null filter : {len(yelp_reviews_raw):>10,}")

# Filter 6: Deduplication on (business_id, text) — catches copy-paste duplicates
before_dedup = len(yelp_reviews_raw)
yelp_reviews_raw.drop_duplicates(subset=["business_id", "text"], keep="first", inplace=True)
print(f"   Reviews removed by deduplication          : {before_dedup - len(yelp_reviews_raw):>10,}")
print(f"   Reviews remaining                         : {len(yelp_reviews_raw):>10,}")

print(f"   Columns now : {list(yelp_reviews_raw.columns)}")

# Validation: star rating distribution
print("\n   ── Yelp star distribution (after Phase 1 cleaning) ──")
print(yelp_reviews_raw["stars"].value_counts().sort_index().to_string())

In [ ]:
tqdm.pandas(desc="Lang-detect Yelp")
yelp_reviews_raw["lang"] = yelp_reviews_raw["text"].progress_apply(detect_language_safe)

yelp_lang_counts = yelp_reviews_raw["lang"].value_counts()
print(f"   Yelp language distribution (top 10):")
print(yelp_lang_counts.head(10).to_string())

yelp_non_en = (yelp_reviews_raw["lang"] != "en").sum()
yelp_non_en_pct   = yelp_non_en / len(yelp_reviews_raw) * 100
yelp_reviews_en = yelp_reviews_raw[yelp_reviews_raw["lang"] == "en"].copy()
print(f"\n   Non-English Yelp reviews removed : {yelp_non_en:>10,}  ({yelp_non_en_pct:.2f}%)")
print(f"   English Yelp reviews retained    : {len(yelp_reviews_en):>10,}")

yelp_reviews_en.drop(columns=["lang"], inplace=True)

In [ ]:
tqdm.pandas(desc="VADER scoring Yelp")
yelp_reviews_en["vader_compound"] = yelp_reviews_en["text"].progress_apply(vader_compound)

print(f"   Yelp VADER compound score statistics:")
print(yelp_reviews_en["vader_compound"].describe().to_string())

In [ ]:
yelp_reviews_en["is_mismatch"] = yelp_reviews_en.apply(
    lambda r: flag_mismatch(r["stars"], r["vader_compound"]), axis=1
)

yelp_mismatch_breakdown = yelp_reviews_en[yelp_reviews_en["is_mismatch"]].apply(
    lambda r: (
        "high_star_neg_text"
        if (r["stars"] >= MISMATCH_HIGH_STAR_THRESH and r["vader_compound"] < MISMATCH_NEG_VADER)
        else "low_star_pos_text"
    ),
    axis=1
).value_counts()

print(f"   ── Yelp mismatch breakdown ──")
print(yelp_mismatch_breakdown.to_string())
print(f"   Total Yelp mismatches flagged : {yelp_reviews_en['is_mismatch'].sum():>8,}")
print(f"   As % of English Yelp reviews  : "
      f"{yelp_reviews_en['is_mismatch'].mean()*100:.2f}%")

yelp_reviews_clean = yelp_reviews_en[~yelp_reviews_en["is_mismatch"]].copy()
yelp_reviews_clean.drop(columns=["is_mismatch"], inplace=True)
print(f"   Yelp reviews after mismatch filter : {len(yelp_reviews_clean):>10,}")

In [ ]:
# ── VADER compound distribution by star rating (Yelp) ──────────────────────
print("   ── Yelp: Mean VADER compound by star rating (post-filter) ──")
print(
    yelp_reviews_clean.groupby("stars")["vader_compound"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_vader", "std": "std_vader", "count": "n_reviews"})
    .to_string()
)

    
# ── Sanity check: no mismatch survivors ────────────────────────────────────
yelp_survivor_mismatches = yelp_reviews_clean.apply(
    lambda r: flag_mismatch(r["stars"], r["vader_compound"]), axis=1
).sum()

print(f"\n   ✅ Yelp mismatch survivors (should be 0)   : {yelp_survivor_mismatches}")

In [ ]:
# ── Save cleaned checkpoints as Parquet (efficient columnar format) ────────
yelp_out   = OUTPUT_DIR / "fyp2_yelp_phase2_clean.parquet"
yelp_reviews_clean.to_parquet(yelp_out,   index=False)

print(f"\n   💾 Yelp checkpoint saved   → {yelp_out}")
print(f"\n✅ Phases 1 & 2 complete. Yelp is ready for Phase 3 (Harmonisation).")

### McAuley Metadata

In [ ]:
sample_meta_file   = google_meta_files[0]
sample_review_file = google_review_files[0]

print(f"Inspecting metadata file : {sample_meta_file.name}")
meta_sample = pd.read_json(sample_meta_file, lines=True, nrows=5)
print(f"   Columns : {list(meta_sample.columns)}")
print(f"   Shape   : {meta_sample.shape}")
display(meta_sample.head(2))

print(f"\nInspecting review file : {sample_review_file.name}")
rev_sample = pd.read_json(sample_review_file, lines=True, nrows=5)
print(f"   Columns : {list(rev_sample.columns)}")
print(f"   Shape   : {rev_sample.shape}")
display(rev_sample.head(2))

# ── Column maps — update these if your files use different names ───────────────
# McAuley Google Local 2021 standard column names:
GOOGLE_META_COLS = {
    "id_col"       : "gmap_id",   # unique place identifier
    "name_col"     : "name",      # business name
    "category_col" : "category",  # list of category strings
    "lat_col"      : "latitude",  # float
    "lon_col"      : "longitude", # float
    "address_col"  : "address",   # string
}

GOOGLE_REV_COLS = {
    "id_col"     : "gmap_id",  # join key
    "rating_col" : "rating",   # numeric 1–5
    "text_col"   : "text",     # review text
    "time_col"   : "time",     # Unix ms timestamp
}

# Confirm expected columns are present
meta_cols_found = {k: v for k, v in GOOGLE_META_COLS.items()
                   if v in meta_sample.columns}
meta_cols_missing = {k: v for k, v in GOOGLE_META_COLS.items()
                     if v not in meta_sample.columns}
rev_cols_found = {k: v for k, v in GOOGLE_REV_COLS.items()
                  if v in rev_sample.columns}
rev_cols_missing = {k: v for k, v in GOOGLE_REV_COLS.items()
                    if v not in rev_sample.columns}

print("── Column map validation ──")
print(f"   Metadata columns found   : {list(meta_cols_found.values())}")
print(f"   Metadata columns MISSING : {list(meta_cols_missing.values())}"
      if meta_cols_missing else "   Metadata columns MISSING : none ✅")
print(f"   Review columns found     : {list(rev_cols_found.values())}")
print(f"   Review columns MISSING   : {list(rev_cols_missing.values())}"
      if rev_cols_missing else "   Review columns MISSING   : none ✅")

if meta_cols_missing or rev_cols_missing:
    print("\n⚠️  Update GOOGLE_META_COLS or GOOGLE_REV_COLS above to match your actual column names.")
else:
    print("\n✅ All expected columns confirmed. Safe to proceed.")

In [ ]:
id_col       = GOOGLE_META_COLS["id_col"]
name_col     = GOOGLE_META_COLS["name_col"]
category_col = GOOGLE_META_COLS["category_col"]
lat_col      = GOOGLE_META_COLS["lat_col"]
lon_col      = GOOGLE_META_COLS["lon_col"]
address_col  = GOOGLE_META_COLS["address_col"]

google_meta_chunks = []
state_meta_counts  = {}

print("=" * 65)
print("PHASE 1A — Google Metadata: Load, Filter & Geo Extraction")
print("=" * 65)
print()

for filepath in tqdm(google_meta_files, desc="Loading metadata state files"):
    state_name = filepath.stem
    state_rows = 0

    try:
        for chunk in pd.read_json(filepath, lines=True, chunksize=CHUNK_SIZE):

            # Step 1: Restaurant domain filter
            if category_col in chunk.columns:
                chunk = chunk[chunk[category_col].apply(is_restaurant_category)]
            else:
                print(f"   ⚠️  '{category_col}' missing in {filepath.name} — skipping category filter")

            if chunk.empty:
                continue

            if name_col in chunk.columns:
                chunk = chunk[
                    ~chunk[name_col].str.contains(
                        r"permanently closed",
                        case=False,
                        na=False
                    )
                ]

            if name_col in chunk.columns:
                chunk = chunk[
                    chunk[name_col].str.contains(r"[A-Za-z]", na=False)
                ]

            if chunk.empty:
                continue

            state_rows += len(chunk)

            # Step 2: Keep id, name, category + geo fields
            keep = [c for c in [
                id_col, name_col, category_col,
                lat_col, lon_col, address_col
            ] if c in chunk.columns]

            google_meta_chunks.append(chunk[keep].copy())

    except Exception as e:
        print(f"   ❌ Error reading {filepath.name}: {e}")
        continue

    state_meta_counts[state_name] = state_rows

google_meta_all = pd.concat(google_meta_chunks, ignore_index=True)

# Remove duplicate gmap_ids (same business can appear in multiple state files near borders)
before_dedup = len(google_meta_all)
google_meta_all.drop_duplicates(subset=[id_col], keep="first", inplace=True)

print(f"\n── Phase 1A Results ──")
print(f"   Raw restaurant businesses loaded : {before_dedup:>10,}")
print(f"   After cross-state deduplication  : {len(google_meta_all):>10,}")
print(f"   Unique gmap_ids                  : {google_meta_all[id_col].nunique():>10,}")
print(f"   Columns retained                 : {list(google_meta_all.columns)}")

# Geo coverage report
for col in [lat_col, lon_col, address_col]:
    if col in google_meta_all.columns:
        cov = google_meta_all[col].notna().mean() * 100
        print(f"   Geo coverage — {col:<12}: {cov:.1f}%")

# Top 5 states by restaurant count
top_states = pd.Series(state_meta_counts).sort_values(ascending=False).head(5)
print(f"\n   Top 5 states by restaurant business count:")
print(top_states.to_string())

In [ ]:
print("=" * 65)
print("PHASE 1B — Chain Filtering: ≥100 Locations Nationally")
print("=" * 65)
print()

# Step 1: Normalize chain names for consistent grouping
google_meta_all["chain_name_norm"] = google_meta_all[name_col].apply(normalize_chain_name)

# Step 2: Count locations per chain across ALL states combined
chain_counts = google_meta_all.groupby("chain_name_norm")[id_col].count()
chain_counts.name = "location_count"

print(f"   Total unique chains (normalized) : {len(chain_counts):>10,}")
print(f"   Chains with ≥{MIN_LOCATIONS} locations         : {(chain_counts >= MIN_LOCATIONS).sum():>10,}")
print(f"   Chains with  <{MIN_LOCATIONS} locations         : {(chain_counts < MIN_LOCATIONS).sum():>10,}")

# Step 3: Filter to qualifying chains
large_chains = chain_counts[chain_counts >= MIN_LOCATIONS]
google_meta_filtered = google_meta_all[
    google_meta_all["chain_name_norm"].isin(large_chains.index)
].copy()
google_meta_filtered = google_meta_filtered.merge(
    large_chains.reset_index(),
    on="chain_name_norm", how="left"
)

GOOGLE_VALID_GMAP_IDS = set(google_meta_filtered[id_col])
print(f"\n   Qualifying gmap_ids              : {len(GOOGLE_VALID_GMAP_IDS):>10,}")

# Step 4: Build business-level geo lookup (one row per gmap_id)
geo_cols_available = [c for c in [lat_col, lon_col, address_col]
                      if c in google_meta_filtered.columns]

google_biz_lookup = (
    google_meta_filtered[[id_col, name_col] + geo_cols_available]
    .drop_duplicates(subset=[id_col])
    .rename(columns={
        id_col     : "gmap_id",
        name_col   : "biz_name",
        lat_col    : "latitude",
        lon_col    : "longitude",
        address_col: "address",
    })
    .copy()
)

print(f"\n   Business geo lookup built        : {len(google_biz_lookup):>10,} unique businesses")
print(f"   Lookup columns                   : {list(google_biz_lookup.columns)}")

# Geo coverage on filtered set
for col in ["latitude", "longitude"]:
    if col in google_biz_lookup.columns:
        cov = google_biz_lookup[col].notna().mean() * 100
        print(f"   Geo coverage (filtered) — {col}: {cov:.1f}%")

print("\n   ── Top 20 qualifying chains by location count ──")
top_chains = (
    google_meta_filtered.groupby("chain_name_norm")["location_count"]
    .first()
    .sort_values(ascending=False)
    .head(20)
)
print(top_chains.to_string())

# Save lookup for use in harmonization
google_biz_lookup.to_parquet(OUTPUT_DIR / "google_biz_lookup.parquet", index=False)
print(f"\n✅ Business geo lookup saved → output/stage1/google_biz_lookup.parquet")

### McAuley Review Data

In [ ]:
# ── Cell 6 — Phase 1C: Load reviews, Phase 1 cleaning, join geo fields ─────────

id_col     = GOOGLE_REV_COLS["id_col"]
rating_col = GOOGLE_REV_COLS["rating_col"]
text_col   = GOOGLE_REV_COLS["text_col"]
time_col   = GOOGLE_REV_COLS["time_col"]

google_review_chunks = []
state_review_counts  = {}

print("=" * 65)
print("PHASE 1C — Google Reviews: Load & Phase 1 Cleaning")
print("=" * 65)
print()

for filepath in tqdm(google_review_files, desc="Loading review state files"):
    state_name  = filepath.stem
    state_kept  = 0
    state_total = 0

    try:
        for chunk in pd.read_json(filepath, lines=True, chunksize=CHUNK_SIZE):
            state_total += len(chunk)

            # Filter 1: qualifying gmap_ids only
            if id_col not in chunk.columns:
                print(f"   ⚠️  '{id_col}' missing in {filepath.name} — skipping")
                continue
            chunk = chunk[chunk[id_col].isin(GOOGLE_VALID_GMAP_IDS)]
            if chunk.empty:
                continue

            # Filter 2: parse timestamp → date, apply date window
            if time_col in chunk.columns:
                chunk["date"] = pd.to_datetime(
                    chunk[time_col], unit="ms", errors="coerce"
                )
            elif "date" in chunk.columns:
                chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
            else:
                chunk["date"] = pd.NaT

            chunk = chunk[
                (chunk["date"] >= DATE_START) & (chunk["date"] <= DATE_END)
            ]
            if chunk.empty:
                continue

            # Filter 3: valid rating range 1–5
            if rating_col not in chunk.columns:
                print(f"   ⚠️  '{rating_col}' missing in {filepath.name} — skipping")
                continue
            chunk = chunk[chunk[rating_col].between(1, 5)]
            if chunk.empty:
                continue

            # Filter 4: drop null and empty text
            if text_col not in chunk.columns:
                print(f"   ⚠️  '{text_col}' missing in {filepath.name} — skipping")
                continue
            text = chunk[text_col].astype("string")
            chunk = chunk[text.notna() & (text.str.strip() != "")]
            if chunk.empty:
                continue

            # Filter 5: lowercase normalization
            chunk[text_col] = chunk[text_col].astype("string").str.lower().str.strip()

            # Filter 6: minimum word count
            chunk = chunk[
                chunk[text_col].astype(str).str.split().str.len() >= MIN_WORD_COUNT
            ]
            if chunk.empty:
                continue

            # Synthesize review_id (Google has no native review id)
            if time_col in chunk.columns:
                chunk["review_id"] = (
                    chunk[id_col].astype(str) + "_" +
                    chunk[time_col].astype(str)
                )
            else:
                chunk["review_id"] = (
                    chunk[id_col].astype(str) + "_" +
                    chunk.index.astype(str)
                )

            # Rename to unified column names
            chunk = chunk.rename(columns={
                rating_col: "stars",
                text_col  : "text",
                id_col    : "gmap_id",
            })

            keep_cols = [c for c in
                         ["review_id", "gmap_id", "stars", "text", "date"]
                         if c in chunk.columns]
            google_review_chunks.append(chunk[keep_cols].copy())
            state_kept += len(chunk)

    except Exception as e:
        print(f"   ❌ Error reading {filepath.name}: {e}")
        continue

    state_review_counts[state_name] = {
        "total": state_total,
        "kept" : state_kept
    }

google_reviews_raw = pd.concat(google_review_chunks, ignore_index=True)

print(f"\n── Phase 1C Results (before deduplication) ──")
print(f"   Reviews loaded (domain+date+null+wordcount) : {len(google_reviews_raw):>10,}")

# Filter 7: deduplication on (gmap_id, text)
before_dedup = len(google_reviews_raw)
google_reviews_raw.drop_duplicates(
    subset=["gmap_id", "text"], keep="first", inplace=True
)
print(f"   Removed by deduplication                    : {before_dedup - len(google_reviews_raw):>10,}")
print(f"   Reviews after deduplication                 : {len(google_reviews_raw):>10,}")

output_parquet_path = "google_reviews_raw_backup.parquet"

print(f"\n💾 Saving {len(google_reviews_raw):,} records to Parquet...")

# Save dataframe
google_reviews_raw.to_parquet(
    output_parquet_path,
    engine="pyarrow",  # Fast and reliable engine
    compression="snappy",  # Good balance of speed and size
    index=False
)

print(f"✅ Successfully saved to: {output_parquet_path}")

In [ ]:
from pandarallel import pandarallel

def detect_language_safe_fixed(text):
    
    if not isinstance(text, str):
        return "unknown"
        
    clean_text = text.strip()
    
    if len(clean_text) < 3:
        return "unknown"
        
    try:
        from langdetect import detect
        from langdetect.detector_factory import DetectorFactory
        
        DetectorFactory.seed = 0 
        
        # 执行检测
        return detect(clean_text[:300])
        
    except Exception:
        return "unknown"

pandarallel.initialize(nb_workers=12, progress_bar=True)

In [ ]:
google_reviews_raw["lang"] = google_reviews_raw["text"].parallel_apply(detect_language_safe_fixed)

lang_dist = google_reviews_raw["lang"].value_counts()
print("   Language distribution (top 15):")
print(lang_dist.head(10).to_string())

non_en_count = (google_reviews_raw["lang"] != "en").sum()
non_en_pct   = non_en_count / len(google_reviews_raw) * 100

google_reviews_en = google_reviews_raw[
    google_reviews_raw["lang"] == "en"
].copy()

print(f"\n   Non-English reviews removed : {non_en_count:>10,}  ({non_en_pct:.2f}%)")
print(f"   English reviews retained    : {len(google_reviews_en):>10,}")

# Drop the lang column — no longer needed
google_reviews_en.drop(columns=["lang"], inplace=True)

# Define output filename
output_parquet_path = "google_reviews_en.parquet"

print(f"\n💾 Saving {len(google_reviews_en):,} records to Parquet...")

# Save dataframe
google_reviews_en.to_parquet(
    output_parquet_path,
    engine="pyarrow",  # Fast and reliable engine
    compression="snappy",  # Good balance of speed and size
    index=False
)

print(f"✅ Successfully saved to: {output_parquet_path}")

In [ ]:
import pandas as pd
from multiprocessing import Pool
from tqdm import tqdm

# Import the worker functions from the file we created in Step 1
import vader_workers

# --- Execution Block ---
if __name__ == "__main__":
    num_workers = 12
    total_rows = len(google_reviews_en)
    
    print(f"Starting parallel processing for {total_rows:,} rows across {num_workers} workers...")

    # Converting to list speeds up process data transfer significantly
    text_data_list = google_reviews_en["text"].tolist()
    
    # Open the process pool using the imported initializer
    with Pool(processes=num_workers, initializer=vader_workers.init_worker) as pool:
        # chunksize=1000 groups data to prevent inter-process communications lag
        results_iterator = pool.imap(vader_workers.process_single_text, text_data_list, chunksize=1000)
        
        # tqdm will now unfreeze and track progress live
        scores = list(tqdm(results_iterator, total=total_rows, desc="VADER Scoring Progress"))

    # Map the final list back into your main DataFrame
    google_reviews_en["vader_compound"] = scores
    print("VADER scoring complete!")

print("   VADER compound score statistics:")
print(google_reviews_en["vader_compound"].describe().to_string())

print("\n   Mean VADER compound by star rating:")
print(
    google_reviews_en.groupby("stars")["vader_compound"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_vader", "std": "std_vader", "count": "n"})
    .to_string()
)

In [ ]:
print(f"   ── Google mismatch breakdown ──")

# 1. Vectorized Flagging (Replaces the first .apply)
# Assigns True if the row matches either high-star-neg-text OR low-star-pos-text
high_star_neg_mask = (google_reviews_en["stars"] >= MISMATCH_HIGH_STAR_THRESH) & (google_reviews_en["vader_compound"] < MISMATCH_NEG_VADER)
low_star_pos_mask = (google_reviews_en["stars"] <= MISMATCH_LOW_STAR_THRESH) & (google_reviews_en["vader_compound"] > MISMATCH_POS_VADER)

google_reviews_en["is_mismatch"] = high_star_neg_mask | low_star_pos_mask

# 2. Vectorized Breakdown Classification (Replaces the second .apply)
# We only calculate categories for rows where is_mismatch is True
mismatch_df = google_reviews_en[google_reviews_en["is_mismatch"]]

mismatch_categories = np.where(
    mismatch_df["stars"] >= MISMATCH_HIGH_STAR_THRESH, 
    "high_star_neg_text", 
    "low_star_pos_text"
)

# Get the count breakdown
google_mismatch_breakdown = pd.Series(mismatch_categories).value_counts()

# 3. Print Results (Fixed the print statement label from 'Yelp' to 'Google')
print(google_mismatch_breakdown.to_string())
print(f"   Total Google mismatches flagged : {google_reviews_en['is_mismatch'].sum():>8,}")
print(f"   As % of English Google reviews  : "
      f"{google_reviews_en['is_mismatch'].mean()*100:.2f}%")

# 4. Clean Dataframe Creation
google_reviews_clean = google_reviews_en[~google_reviews_en["is_mismatch"]].copy()
google_reviews_clean.drop(columns=["is_mismatch"], inplace=True)
print(f"   Google reviews after mismatch filter : {len(google_reviews_clean):>10,}")


In [ ]:
# ──  VADER compound distribution by star rating (Google) ───────────────────
print("\n   ── Google: Mean VADER compound by star rating (post-filter) ──")
if not google_reviews_clean.empty:
    print(
        google_reviews_clean.groupby("stars")["vader_compound"]
        .agg(["mean", "std", "count"])
        .rename(columns={"mean": "mean_vader", "std": "std_vader", "count": "n_reviews"})
        .to_string()
)
else:
    print("   No Google reviews available to compute statistics.")

# ──  Sanity check: no mismatch survivors ────────────────────────────────────
google_survivor_mismatches = google_reviews_clean.apply(
    lambda r: flag_mismatch(r["stars"], r["vader_compound"]), axis=1
).sum()
print(f"   ✅ Google mismatch survivors (should be 0) : {google_survivor_mismatches}")

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("output/stage1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

google_out = OUTPUT_DIR / "google_phase2_clean.parquet"
google_reviews_clean.to_parquet(google_out, index=False)

print(f"   💾 Google checkpoint saved → {google_out}")
print(f"\n✅ Phases 1 & 2 complete. Ready for Phase 3 (Harmonisation).")

## 2nd Filtration

### Early Setup

In [ ]:
# ============================================================
# Install dependencies
# ============================================================
# !pip install requests beautifulsoup4 lxml pandas nltk -q
# !pip install playwright -q
# !playwright install chromium --with-deps
# %pip install ddgs

In [ ]:
# ============================================================
# Imports
# ============================================================
import re
import time
import random
import threading
import queue
import urllib.parse as urlparse
from urllib.robotparser import RobotFileParser
from typing import Optional

import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# ============================================================
# Step 1 — Isolate whether this is DDG-specific or a general
# network issue, and whether it's a fresh problem or persistent
# ============================================================
test_urls = [
    "https://html.duckduckgo.com/html/",   # the one that's failing
    "https://www.bing.com",                 # different domain, sanity check
    "https://www.google.com",               # different domain, sanity check
    "https://api.github.com",               # unrelated domain, sanity check
]

for url in test_urls:
    try:
        resp = requests.get(url, timeout=8)
        print(f"{url:40s} -> OK, status {resp.status_code}")
    except requests.exceptions.ConnectTimeout:
        print(f"{url:40s} -> CONNECT TIMEOUT")
    except requests.exceptions.RequestException as e:
        print(f"{url:40s} -> {type(e).__name__}: {e}")

In [ ]:
# ============================================================
# Global config
# ============================================================
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0 Safari/537.36"
}

BROWSER_HEADERS = {
    "User-Agent": HEADERS["User-Agent"],
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

REQUEST_TIMEOUT = 10
DELAY_RANGE = (1.5, 3.0)

ABOUT_KEYWORDS = [
    "about", "about-us", "our-story", "our story", "who-we-are",
    "vision", "mission", "vision-mission", "our-mission", "brand",
    "philosophy", "our-values", "purpose", "heritage"
]

BLOCKED_HOSTS = ("yelp.", "facebook.", "instagram.", "wikipedia.",
                  "tripadvisor.", "linkedin.", "twitter.", "x.com",
                  "doordash.", "ubereats.", "grubhub.", "youtube.",
                  "reddit.", "glassdoor.", "indeed.",
                  "duckduckgo.com", "duckduckgo.", "bing.com",
                  "google.com", "microsoft.com")

BLOCKED_HOST = ("yelp.", "facebook.", "instagram.", "wikipedia.",
                "tripadvisor.", "linkedin.", "twitter.", "x.com",
                "doordash.", "ubereats.", "grubhub.", "youtube.",
                "reddit.", "glassdoor.", "indeed.")

DOMAIN_OVERRIDES = {
    # "bojangles": "bojangles.com",
}

def _sleep():
    time.sleep(random.uniform(*DELAY_RANGE))

### identify unique business name and resolve the domain url

In [ ]:
import pandas as pd
yelp = pd.read_parquet('filtered_yelp_biz_lookup.parquet')

google = pd.read_parquet('filtered_google_biz_lookup.parquet')

# Combine unique names from both dataframes while eliminating spacing and casing anomalies
unique_names_set = set(yelp['name'].str.strip().str.title().dropna().unique()).union(
    set(google['biz_name'].str.strip().str.title().dropna().unique())
)

# Convert to an organized, alphabetic Python list ready for scraping scripts
brands = sorted(list(unique_names_set))
print(f"Stored {len(brands)} unique clean restaurant brands.")

df = pd.DataFrame({"name": brands})
for col in ["domain", "homepage_status", "robots_allowed", "js_rendered_site",
            "scrapable", "about_url", "vision_url", "about_text", "vision_text", "notes"]:
    df[col] = None

df.info()

In [ ]:
def resolve_domain(brand_name: str, max_retries: int = 3) -> Optional[str]:
    clean_name = re.sub(r"[’'`]", "", brand_name)
    clean_name = re.sub(r"\s+", " ", clean_name).strip()
    query = f"{clean_name} official website"

    for attempt in range(max_retries):
        try:
            resp = requests.get("https://html.duckduckgo.com/html/",
                                 params={"q": query}, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue

        soup = BeautifulSoup(resp.text, "lxml")
        links = soup.select("a.result__a")
        if not links:
            time.sleep(3 + 2 ** attempt)
            continue

        for a in links:
            href = a.get("href", "")
            qs = urlparse.parse_qs((urlparse.urlparse(href)).query)
            real_url = qs.get("uddg", [href])[0]
            real_domain = urlparse.urlparse(real_url).netloc.lower()
            if real_domain and not any(b in real_domain for b in BLOCKED_HOSTS):
                return real_domain
        return None

    return None


def resolve_domain_bing(brand_name: str) -> Optional[str]:
    clean_name = re.sub(r"[’'`]", "", brand_name)
    query = f"{clean_name} official website"
    try:
        resp = requests.get("https://www.bing.com/search", params={"q": query},
                             headers=HEADERS, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
    except requests.RequestException:
        return None

    soup = BeautifulSoup(resp.text, "lxml")
    for a in soup.select("li.b_algo h2 a, li.b_algo a"):
        domain = urlparse.urlparse(a.get("href", "")).netloc.lower()
        if domain and not any(b in domain for b in BLOCKED_HOSTS):
            return domain
    return None


def guess_domain_directly(brand_name: str) -> Optional[str]:
    slug = re.sub(r"[’'`.,&]", "", brand_name.lower())
    slug = re.sub(r"\b(restaurant|grill|bar|kitchen|cafe|inc)\b", "", slug)
    slug = re.sub(r"[^a-z0-9]+", "", slug).strip()

    for domain in (f"{slug}.com", f"{slug}usa.com", f"eat{slug}.com"):
        try:
            resp = requests.head(f"https://{domain}", headers=HEADERS, timeout=6, allow_redirects=True)
            if resp.status_code < 400:
                return urlparse.urlparse(resp.url).netloc
        except requests.RequestException:
            continue
    return None


def resolve_domain_full(brand_name: str) -> Optional[str]:
    key = brand_name.lower().strip()
    if key in DOMAIN_OVERRIDES:
        return DOMAIN_OVERRIDES[key]
    return (resolve_domain(brand_name)
            or resolve_domain_bing(brand_name)
            or guess_domain_directly(brand_name))

In [ ]:
to_resolve = df.loc[df['domain'].isna(), 'name'].tolist()
print(f"Resolving {len(to_resolve)} domains...")

for name in to_resolve:
    domain = resolve_domain_full(name)
    df.loc[df['name'] == name, 'domain'] = domain
    if not domain:
        df.loc[df['name'] == name, 'notes'] = "Could not resolve an official domain via search."
    print(f"  {name} -> {domain}")
    _sleep()

print(f"\nDomains resolved: {df['domain'].notna().sum()} / {len(df)}")

df.to_csv("finalv1.csv", index=False)
print(df.info())

## To be continue...